# EternityQuant — Colab ML 训练

在 Colab 的免费 T4 GPU 上训练 EternityQuant 的量化选股模型。

## 支持算法
- **LightGBM** (CPU) — 基线模型，qlib Alpha158 特征
- **MLP** (CUDA) — 自写全连接网络，158→512→256→128→1
- **GRU** (CUDA) — 自写时序模型，6×26 时序重塑，量化选股最佳
- **LSTM** (CUDA) — 与 GRU 同架构，3 门替代 2 门
- **DeepLOB** (CUDA) — CNN+BiLSTM+Attention，顶会论文复现
- **TFT** (CUDA) — Temporal Fusion Transformer，Google 多时间跨度预测

## v0.20 关键升级
- **统一数据目录**：所有数据集中到 `~/.eternityquant/data/{a,hk,us}/` 下
- **权威标准化处理器**：对标 qlib 官方 benchmarks，特征走 `RobustZScoreNorm(clip_outlier=True)`，标签走 `CSZScoreNorm`/`CSRankNorm`（MAD 抗异常值）

## 多卡 GPU 支持
Colab 单台只有 1 张 T4，但代码支持 `--gpus "0,1,2,3"` 参数，
在租用多 GPU 云服务器（如 A100×4、4090×2）时自动启用 `nn.DataParallel` 并行训练。

## 输出
- 训练好的模型文件（`.pkl`）→ 下载回本地，用 `eq ml register/activate` 导入
- 训练指标（IC、ICIR、Rank IC）

---

## 1. 环境准备

安装依赖 + 克隆 EternityQuant 代码。

In [ ]:
# @title 安装依赖
import sys, os, subprocess

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(pkgs))

def _pip_ok(*pkgs):
    """安装包，返回是否成功（不抛异常）"""
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(pkgs),
                              stderr=subprocess.DEVNULL)
        return True
    except subprocess.CalledProcessError:
        return False

# 基础依赖
_pip("numpy", "pandas", "typer", "python-dotenv")

# qlib — PyPI 版 0.9.7 有两个 bug:
#   1. LocalDatasetProvider 双重复值 inst_processors (issue #1949)
#   2. Corr._load_internal 的 series_right 为空导致广播失败
# 两个 bug 均在 GitHub master 中修复，故从源码安装
qlib_ok = False
print("安装 qlib（从 GitHub 源码，约 2 分钟）...", flush=True)
import subprocess
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/microsoft/qlib.git", "--no-build-isolation"],
        stderr=subprocess.DEVNULL)
    qlib_ok = True
    print("  ✓ qlib GitHub 源码安装成功")
except subprocess.CalledProcessError:
    print("  ✗ qlib 源码安装失败，降级到 PyPI 版")
    # 降级：PyPI 版用 joblib==1.2.0 绕开 issue #1949
    _pip("joblib==1.2.0")
    if _pip_ok("pyqlib>=0.9", "--only-binary=:all:"):
        qlib_ok = True
        print("  ✓ qlib PyPI wheel 安装成功")
    else:
        print("  ✗ qlib 安装失败（Cython/numpy 版本不兼容）")
        print("    A 股 qlib 训练不可用，但港股训练不受影响（hk_market 不依赖 qlib）")
        print("    如需 A 股训练，手动运行: pip install pyqlib --no-build-isolation")

# 验证 qlib 可导入
if qlib_ok:
    try:
        import qlib
        print(f"  qlib {qlib.__version__} 可用")
    except ImportError as e:
        qlib_ok = False
        print(f"  ✗ qlib 导入失败: {e}")

# LightGBM
_pip("lightgbm>=4.0")

# akshare（成分股列表获取）
_pip("akshare>=1.10")

# PyTorch (Colab 预装 CUDA 版，只需确认)
import torch
print(f"\nPyTorch {torch.__version__}  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print("\n依赖安装完成 ✓")
print(f"qlib 可用: {qlib_ok}  (A 股训练需要，港股训练不需要)")

In [ ]:
# @title 克隆 EternityQuant
import os
REPO_URL = "https://github.com/Eternity231/EternityQuant.git"
PROJECT_DIR = "/content/EternityQuant"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull

# 关键：设置 EQ_HOME 指向用户主目录 ~/.eternityquant
# 否则数据和模型会落到项目内 .eternityquant/，与 tar 包解压路径不一致
os.environ["EQ_HOME"] = os.path.expanduser("~/.eternityquant")
os.makedirs(os.environ["EQ_HOME"], exist_ok=True)
print(f"EQ_HOME = {os.environ['EQ_HOME']}")

os.chdir(PROJECT_DIR)
!pip install -e . --quiet
print(f"工作目录: {os.getcwd()}")

## 2. 准备 qlib 数据

### 统一数据目录
所有数据集中到项目内 `.eternityquant/data/a/qlib_cn_data/` 下。
在 Colab 上路径为 `/content/EternityQuant/.eternityquant/data/a/qlib_cn_data/`。

### 方案 A：上传本地打包的 `cn_data.tar.gz`（推荐，快）
在本地跑 `eq ml update-data --start 2015-01-01 --universe csi300 --workers 5` 下载 A 股数据，
然后用 `tar czf cn_data.tar.gz -C ~/.eternityquant/data/a/qlib_cn_data .` 打包上传到 Google Drive。

### 方案 B：在 Colab 中在线拉取（慢，约 30 分钟）
用腾讯 API 直接拉取 A 股数据，自动落到项目内统一目录。

In [ ]:
# @title 挂载 Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# @title 方案 A：从 Google Drive 解压 cn_data.tar.gz
import os
from pathlib import Path

# 统一目录：项目内 .eternityquant/data/a/qlib_cn_data/
PROJECT_DIR = "/content/EternityQuant"
QLIB_TARGET = Path(PROJECT_DIR) / ".eternityquant" / "data" / "a" / "qlib_cn_data"
QLIB_TARGET.mkdir(parents=True, exist_ok=True)

# 如果 cn_data.tar.gz 在 Google Drive 中，从 Drive 复制
DRIVE_TAR = "/content/drive/MyDrive/qlib_data/cn_data.tar.gz"
if os.path.exists(DRIVE_TAR):
    !tar xzf "{DRIVE_TAR}" -C {QLIB_TARGET}/
    print(f"已从 Drive 解压到 {QLIB_TARGET}")
elif os.path.exists("cn_data.tar.gz"):
    !tar xzf cn_data.tar.gz -C {QLIB_TARGET}/
    print(f"已从当前目录解压到 {QLIB_TARGET}")
else:
    print("未找到 cn_data.tar.gz，请上传或切换到方案 B")

# 验证
cal = QLIB_TARGET / "calendars" / "day.txt"
if cal.exists():
    print(f"✓ 数据就绪: {cal}")
    n_features = len(list((QLIB_TARGET / "features").iterdir())) if (QLIB_TARGET / "features").exists() else 0
    print(f"  特征目录: {n_features} 只股票")
else:
    print("✗ 数据未找到")

## 3. 港股数据（可选）

如果要在 Colab 训练港股模型，把本地打包的 `hk_data.tar.gz` 上传到 Google Drive 的 `MyDrive/qlib_data/` 目录下。

本地打包命令：
```bash
tar czf hk_data.tar.gz -C ~/.eternityquant/data/hk .
```

In [ ]:
# @title 解压 hk_data.tar.gz 到统一港股目录
import os
from pathlib import Path

# 统一目录：项目内 .eternityquant/data/hk/
PROJECT_DIR = "/content/EternityQuant"
HK_TARGET = Path(PROJECT_DIR) / ".eternityquant" / "data" / "hk"
HK_TARGET.mkdir(parents=True, exist_ok=True)

DRIVE_HK_TAR = "/content/drive/MyDrive/qlib_data/hk_data.tar.gz"
if os.path.exists(DRIVE_HK_TAR):
    !tar xzf "{DRIVE_HK_TAR}" -C {HK_TARGET}/
    print(f"✓ 已从 Drive 解压到 {HK_TARGET}")
elif os.path.exists("hk_data.tar.gz"):
    !tar xzf hk_data.tar.gz -C {HK_TARGET}/
    print(f"✓ 已从当前目录解压到 {HK_TARGET}")
else:
    print("未找到 hk_data.tar.gz，跳过港股数据（仅训练 A 股可忽略此单元格）")

# 验证港股数据
for sub in ("daily", "5m", "1m"):
    d = HK_TARGET / sub
    if d.exists():
        n = len(list(d.iterdir()))
        print(f"  hk/{sub}: {n} 个文件")
    else:
        print(f"  hk/{sub}: 未找到")

In [ ]:
# @title 方案 B：在线拉取 A 股数据（腾讯 API，自动落到项目内 .eternityquant）
import os
from pathlib import Path

PROJECT_DIR = "/content/EternityQuant"
QLIB_DATA_DIR = Path(PROJECT_DIR) / ".eternityquant" / "data" / "a" / "qlib_cn_data"

if not QLIB_DATA_DIR.exists() or not (QLIB_DATA_DIR / "calendars").exists():
    os.chdir("/content/EternityQuant")
    from eq.strategy.factors.ml_data_updater import update_qlib_data
    result = update_qlib_data(
        start="2015-01-01",  # 拉近 10 年数据
        universe="csi300",
        workers=5,
        verbose=True,
    )
    print(f"数据更新完成: {result}")
else:
    print(f"qlib 数据已存在: {QLIB_DATA_DIR}")

# 验证
cal = QLIB_DATA_DIR / "calendars" / "day.txt"
if cal.exists():
    with open(cal) as f:
        lines = f.read().strip().split("\n")
    print(f"✓ 日历: {len(lines)} 个交易日  {lines[0]} ~ {lines[-1]}")
feat_dir = QLIB_DATA_DIR / "features"
if feat_dir.exists():
    n = len(list(feat_dir.iterdir()))
    print(f"✓ 特征: {n} 只股票")
else:
    print("✗ 特征目录未找到")

In [ ]:
# @title 迁移旧目录数据到统一目录（如有旧数据）
import os
os.chdir("/content/EternityQuant")
from eq.data.paths import migrate_legacy_data_layout
result = migrate_legacy_data_layout(verbose=True)
print(f"\n迁移完成: 复制 {len(result['copied'])} 项，跳过 {len(result['skipped'])} 项")
print("旧目录保留，可手动删除。统一目录结构：")
print("  ~/.eternityquant/data/a/qlib_cn_data/  (A 股 qlib)")
print("  ~/.eternityquant/data/hk/              (港股)")
print("  ~/.eternityquant/data/us/              (美股)")

## 3. 训练模型

### v0.20 权威标准化处理器链
对标 qlib 官方 benchmarks（GRU/ALSTM/LightGBM）同款配置：

| 路径 | 特征处理器 (infer) | 标签处理器 (learn) |
|------|-------------------|-------------------|
| `train()` (LightGBM) | ProcessInf → **RobustZScoreNorm(clip)** → Fillna | DropnaLabel → **CSZScoreNorm** |
| `train_torch()` (RNN/DeepLOB/TFT) | 同上 | DropnaLabel → **CSRankNorm** |

**关键升级**：`RobustZScoreNorm` 用 MAD（中位绝对偏差）替代 std，抗异常值；`CSRankNorm` 横截面排序归一化，免疫异常值。处理器参数在 `fit_start_time`/`fit_end_end` 范围内学习，绝无数据泄露。

### 3.1 LightGBM (CPU 基线)

In [ ]:
# @title 训练 LightGBM
import sys, os
os.chdir("/content/EternityQuant")
sys.path.insert(0, ".")

from eq.strategy.factors.ml_workflow import train

# train() 专走 LightGBM 路径（device 在内部固定 cpu/gpu）
result = train(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2024-12-31",
    valid_start="2025-01-01",
    valid_end="2026-07-14",
    horizon=5,
    algo="lightgbm",
    device="cpu",
    name="colab_lgbm_csi300_h5",
)
print(f"\n训练完成:")
print(f"  Model ID: {result['model_id']}")
print(f"  IC: {result['metrics']['ic']:.4f}")
print(f"  模型文件: {result['model_path']}")

### 3.2 MLP (CUDA)

In [ ]:
# @title 训练 MLP
import sys, os
os.chdir("/content/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2024-12-31",
    valid_start="2025-01-01",
    valid_end="2026-07-14",
    horizon=5,
    algo="mlp",
    device="cuda",
    name="colab_mlp_csi300_h5",
)
print(f"\n训练完成:")
print(f"  Model ID: {result['model_id']}")
print(f"  IC: {result['metrics']['ic']:.4f}")
print(f"  模型文件: {result['model_path']}")

### 3.3 GRU (CUDA) — 推荐，量化选股最佳

In [ ]:
# @title 训练 GRU (T4 优化参数)
import sys, os
os.chdir("/content/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

# T4 16GB 优化参数: batch_size=1024, hidden_size=128, num_layers=2
result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2024-12-31",
    valid_start="2025-01-01",
    valid_end="2026-07-14",
    horizon=5,
    algo="gru",
    device="cuda",
    hidden_size=128,
    num_layers=2,
    batch_size=1024,
    name="colab_gru_csi300_h5",
)
print(f"\n训练完成:")
print(f"  Model ID: {result['model_id']}")
print(f"  IC: {result['metrics']['ic']:.4f}")
print(f"  Epochs: {result['metrics']['epochs']}")
print(f"  模型文件: {result['model_path']}")

### 3.4 LSTM (CUDA)

In [ ]:
# @title 训练 LSTM (T4 优化参数)
import sys, os
os.chdir("/content/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2024-12-31",
    valid_start="2025-01-01",
    valid_end="2026-07-14",
    horizon=5,
    algo="lstm",
    device="cuda",
    hidden_size=128,
    num_layers=2,
    batch_size=1024,
    name="colab_lstm_csi300_h5",
)
print(f"\n训练完成:")
print(f"  Model ID: {result['model_id']}")
print(f"  IC: {result['metrics']['ic']:.4f}")
print(f"  模型文件: {result['model_path']}")

### 3.5 DeepLOB (CUDA) — CNN+BiLSTM+Attention

[Zhang et al., 2019] 针对金融微观结构设计的专用架构。
卷积层提取空间特征，BiLSTM 建模时序，注意力机制加权。

In [ ]:
# @title 训练 DeepLOB (T4 优化参数)
import sys, os
os.chdir("/content/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

# T4 16GB 优化参数: batch_size=256, seq_len=120, hidden=64, dropout=0.3
result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2024-12-31",
    valid_start="2025-01-01",
    valid_end="2026-07-14",
    horizon=5,
    algo="deeplob",
    device="cuda",
    optimizer="adamw",
    loss_type="sharpe",
    batch_size=256,
    seq_len=120,
    dropout=0.3,
    name="colab_deeplob_csi300_h5",
)
print(f"\n训练完成:")
print(f"  Model ID: {result['model_id']}")
print(f"  IC: {result['metrics']['ic']:.4f}")
print(f"  模型文件: {result['model_path']}")

### 3.6 TFT (CUDA) — Temporal Fusion Transformer

[Lim et al., 2019] Google 多时间跨度预测模型。
多头注意力+GRN+位置编码，目前中低频时序最先进模型之一。

In [ ]:
# @title 训练 TFT (T4 优化参数)
import sys, os
os.chdir("/content/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

# T4 16GB 优化参数: batch_size=128, hidden_size=128, num_heads=4, dropout=0.4
result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2024-12-31",
    valid_start="2025-01-01",
    valid_end="2026-07-14",
    horizon=5,
    algo="tft",
    device="cuda",
    optimizer="adamw",
    loss_type="sharpe",
    hidden_size=128,
    num_heads=4,
    batch_size=128,
    dropout=0.4,
    name="colab_tft_csi300_h5",
)
print(f"\n训练完成:")
print(f"  Model ID: {result['model_id']}")
print(f"  IC: {result['metrics']['ic']:.4f}")
print(f"  模型文件: {result['model_path']}")

### 3.7 高级优化器对比

| 优化器 | 核心思想 | 金融优势 |
|--------|---------|---------|
| SAM | 寻找平坦极小值 | 市场环境漂移时仍保持低损失 |
| Lookahead | 双权重：快权探索，慢权稳定 | 降低局部噪音带偏概率 |
| Lion | 只看梯度符号，忽略幅度 | 免疫闪崩等极端异常值 |

In [ ]:
# @title 训练 GRU + SAM 优化器 + 可微夏普比率
import sys, os
os.chdir("/content/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2024-12-31",
    valid_start="2025-01-01",
    valid_end="2026-07-14",
    horizon=5,
    algo="gru",
    device="cuda",
    optimizer="sam",       # Sharpness-Aware Minimization
    loss_type="sharpe",    # 可微夏普比率损失
    name="colab_gru_sam_csi300_h5",
)
print(f"\n训练完成: IC={result['metrics']['ic']:+.4f}")

In [ ]:
# @title 训练 GRU + Lion 优化器
import sys, os
os.chdir("/content/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2024-12-31",
    valid_start="2025-01-01",
    valid_end="2026-07-14",
    horizon=5,
    algo="gru",
    device="cuda",
    optimizer="lion",     # EvoLved Sign Momentum
    loss_type="sharpe",
    name="colab_gru_lion_csi300_h5",
)
print(f"\n训练完成: IC={result['metrics']['ic']:+.4f}")

### 3.8 对抗训练 + 特征正交化

FGSM 对抗训练 + 特征正交化去 Beta，提升模型在实盘异常行情中的鲁棒性。

In [ ]:
# @title 训练 TFT + 对抗训练 + 特征正交化
import sys, os
os.chdir("/content/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2024-12-31",
    valid_start="2025-01-01",
    valid_end="2026-07-14",
    horizon=5,
    algo="tft",
    device="cuda",
    optimizer="adamw",
    loss_type="sharpe",
    adversarial=True,       # FGSM 对抗训练
    orthogonalize=True,     # 特征正交化去 Beta
    name="colab_tft_adv_orth_csi300_h5",
)
print(f"\n训练完成: IC={result['metrics']['ic']:+.4f}")

### 3.9 多卡并行训练（多 GPU 服务器专用）

Colab 只有 1 张 T4，但如果租用多 GPU 云服务器（如 A100×4、4090×2），
可以用 `gpu_ids` 参数启用 `nn.DataParallel` 自动并行：

```python
# 双卡训练 TFT
train_torch(universe="csi300", horizon=5, algo="tft", device="cuda",
            gpu_ids="0,1", name="multigpu_tft")

# 四卡训练 DeepLOB
train_torch(universe="csi300", horizon=5, algo="deeplob", device="cuda",
            gpu_ids="0,1,2,3", name="multigpu_deeplob")
```

### 3.10 超参搜索（自动找最佳 LSTM/GRU 参数）

In [ ]:
# @title GRU 超参网格搜索
import sys, os
os.chdir("/content/EternityQuant")

from eq.strategy.factors.ml_workflow import search_lstm

results = search_lstm(
    universe="csi300",
    horizon=5,
    device="cuda",
    fast=True,        # 快速模式：每组合 max_steps=50
    auto_train=True,  # 搜索完自动用最佳参数全量训练
    algo="gru",      # 可选 "lstm" 或 "gru"
)

# 打印 Top3 结果
print(f"\n搜索完成 {len(results)} 组合")
for i, r in enumerate(results[:3]):
    print(f"  #{i+1}: hidden={r['hidden_size']} layers={r['num_layers']} "
          f"lr={r['lr']} batch={r['batch_size']}  IC={r['ic']:+.4f}")

### 3.11 港股模型训练（GRU 多频率）

港股训练不走 qlib，用 `eq.data.hk_market` 的自写特征 + `_SimpleSeqModel`。

支持三个频率分别训练：
- `train_hk()` — 日线，horizon=5（5 日后收益）
- `train_hk_minute(freq="5m")` — 5 分钟线，horizon=30（2.5 小时后）
- `train_hk_minute(freq="1m")` — 1 分钟线，horizon=30（30 分钟后）

训练后可用 `predict_hk_ensemble()` 做多频率集成预测。

In [ ]:
# @title 训练港股日线 GRU
import sys, os
os.chdir("/content/EternityQuant")

from eq.data.hk_market import train_hk

result = train_hk(
    top_n=100,           # 取港股前 100 只
    horizon=5,           # 预测 5 日后收益
    hidden_size=128,
    num_layers=2,
    cell_type="gru",
    batch_size=512,
    max_steps=200,
    device="cuda",
    dropout=0.3,
    walk_forward=True,   # 滚动前向验证
    name="colab_hk_gru_h5",
)
print(f"\n港股日线训练完成:")
print(f"  IC: {result['ic']:+.4f}")
print(f"  模型: {result['model_path']}")
print(f"  样本: {result['train_samples']}")

In [ ]:
# @title 训练港股 5 分钟线 GRU
import sys, os
os.chdir("/content/EternityQuant")

from eq.data.hk_market import train_hk_minute

result = train_hk_minute(
    top_n=50,
    freq="5m",
    horizon=30,          # 30 根 5m = 2.5 小时后
    hidden_size=128,
    num_layers=2,
    cell_type="gru",
    batch_size=512,
    max_steps=200,
    device="cuda",
    dropout=0.3,
    walk_forward=True,
    name="colab_hk_gru_5m_h30",
)
print(f"\n港股 5m 训练完成: IC={result['ic']:+.4f}  {result['model_path']}")

In [ ]:
# @title 训练港股 1 分钟线 GRU
import sys, os
os.chdir("/content/EternityQuant")

from eq.data.hk_market import train_hk_minute

result = train_hk_minute(
    top_n=50,
    freq="1m",
    horizon=30,          # 30 根 1m = 30 分钟后
    hidden_size=128,
    num_layers=2,
    cell_type="gru",
    batch_size=512,
    max_steps=200,
    device="cuda",
    dropout=0.3,
    walk_forward=True,
    name="colab_hk_gru_1m_h30",
)
print(f"\n港股 1m 训练完成: IC={result['ic']:+.4f}  {result['model_path']}")

In [ ]:
# @title 港股多频率集成预测 Top 10
import sys, os
os.chdir("/content/EternityQuant")

from eq.data.hk_market import predict_hk_ensemble
from pathlib import Path

models_dir = Path.home() / ".eternityquant" / "data" / "hk" / "models"
daily_pkl = str(models_dir / "colab_hk_gru_h5.pkl")
five_m_pkl = str(models_dir / "colab_hk_gru_5m_h30.pkl")
one_m_pkl = str(models_dir / "colab_hk_gru_1m_h30.pkl")

top_df = predict_hk_ensemble(
    model_daily=daily_pkl,
    model_5m=five_m_pkl if os.path.exists(five_m_pkl) else None,
    model_1m=one_m_pkl if os.path.exists(one_m_pkl) else None,
    top_n=10,
    weights={"daily": 0.5, "5m": 0.3, "1m": 0.2},
)
print("\n港股多频率集成预测 Top 10:")
print(top_df.to_string(index=False))

## 4. 导出模型回本地

训练好的模型文件在 `~/.eternityquant/ml_models/` 目录下（Colab 上即 `/root/.eternityquant/ml_models/`）。
将其下载到本地，然后用 `eq ml register` 和 `eq ml activate` 导入。

In [ ]:
# @title 列出所有训练好的模型
import os
from pathlib import Path
models_dir = Path.home() / ".eternityquant" / "ml_models"
if models_dir.exists():
    for f in sorted(models_dir.iterdir()):
        size = f.stat().st_size
        print(f"  {f.name:45s}  {size/1024:.1f} KB")
else:
    print("未找到模型文件")

In [ ]:
# @title 打包下载所有模型到本地
import os
from pathlib import Path
models_dir = Path.home() / ".eternityquant" / "ml_models"
if models_dir.exists() and any(models_dir.iterdir()):
    !tar czf /content/eternityquant_models.tar.gz -C {models_dir} .
    from google.colab import files
    files.download("/content/eternityquant_models.tar.gz")
else:
    print("没有模型文件可下载")

## 5. 回本地导入模型

```bash
# 解压模型文件
tar xzf eternityquant_models.tar.gz -C ~/.eternityquant/ml_models/

# 登记模型
eq ml register --name "colab_gru_csi300_h5" --universe csi300 \
    --features "Alpha158" --algo gru --horizon 5 \
    --train-period "2015-01-01~2024-12-31" \
    --model-path ~/.eternityquant/ml_models/torch_gru_csi300_5d.pkl \
    --metrics '{"ic": 0.15}' \
    --notes "Colab T4 GPU 训练"

# 激活模型
eq ml activate <model_id>

# 批量预测今日 Top 10
eq ml predict-batch <model_id> --top 10
```

---

## 附：Colab GPU 配置信息

In [ ]:
# @title GPU 信息
import torch, psutil, platform
print(f"系统: {platform.platform()}")
print(f"CPU: {psutil.cpu_count()} 核")
print(f"内存: {psutil.virtual_memory().total / 1e9:.1f} GB")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA 版本: {torch.version.cuda}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("GPU: 不可用（将使用 CPU）")